In [47]:
# =============================================================================
#  UTILITAS BERSAMA
#  Dipakai oleh tahap training, inferensi, dan export hasil
# =============================================================================

import re
import warnings
import numpy as np
import pandas as pd
import joblib
import time
import os
from joblib import Parallel, delayed
from rapidfuzz import fuzz
from IPython.display import display
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")

MODEL_PATH = "model_sqli_nb.pkl"

STOPWORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for",
    "of", "and", "with", "by", "as", "be", "was", "are", "were",
    "this", "that", "have", "has", "had", "do", "does", "did",
    "but", "so", "if", "then", "than", "its", "into", "from",
    "there", "their", "they", "will", "would", "could", "should"
}

SQL_KEYWORDS = {
    "select", "from", "where", "and", "or", "not", "is", "in",
    "like", "union", "insert", "update", "delete", "drop", "create",
    "table", "into", "values", "order", "by", "group", "having",
    "join", "on", "null", "true", "false", "case", "when", "then",
    "else", "end", "limit", "offset", "between", "exists", "all",
    "distinct", "count", "sum", "max", "min", "avg", "sleep",
    "benchmark", "char", "concat", "substring", "load_file",
    "outfile", "exec", "execute", "cast", "convert", "if"
}

In [48]:
# =============================================================================
#  1. PRE-COMPILE REGEX PATTERNS (Cukup di-run sekali di luar fungsi)
# =============================================================================
# Menggunakan [^'] jauh lebih cepat daripada (.*?) karena menghilangkan backtracking
REGEX_SINGLE_QUOTE = re.compile(r"'[^']*'")
REGEX_DOUBLE_QUOTE = re.compile(r'"[^"]*"')
REGEX_DIGITS = re.compile(r'\d+')

# Gabungkan tokenisasi menjadi satu pattern solid
REGEX_TOKENS = re.compile(r"[a-z0-9_]+|--|/\*|\*/|'|\"|\(|\)|=|<|>|;|#|,|\*|\+|-|%")


def preprocess_text(text):
    """
    Fungsi preprocessing teks versi optimasi tinggi.
    Menghapus redundansi proses dan mempercepat eksekusi regex & loop.
    """
    if not isinstance(text, str):
        return ""
    
    # 2. Eksekusi Regex yang sudah di-compile
    text = REGEX_SINGLE_QUOTE.sub("'str'", text)
    text = REGEX_DOUBLE_QUOTE.sub('"str"', text)
    text = REGEX_DIGITS.sub('0', text)

    # 3. Tokenisasi cepat
    tokens = REGEX_TOKENS.findall(text)

    # 4. OPTIMASI LOOP: List Comprehension berbasis C-level
    # Logika ekkuivalen: Simpan jika dia adalah KEYWORD SQL ATAU dia BUKAN termasuk STOPWORDS
    filtered = [t for t in tokens if (t in SQL_KEYWORDS or t not in STOPWORDS)]

    return " ".join(filtered)

# Fungsi pembantu untuk memeriksa leakage per baris data test
def cek_kebocoran_per_baris(t_raw_query, train_list):
    for tr_raw_query in train_list:
        if fuzz.ratio(t_raw_query, tr_raw_query) > 90:
            return True # Bocor
    return False # Aman

In [49]:
# =============================================================================
#  TAHAP 1 — MEMUAT DATASET (VERSI OPTIMAL & SUPER CEPAT)
# =============================================================================

print("=" * 65)
print("  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES")
print("=" * 65)

DATASET_PATH = "rbsqli_dataset.csv"

# OPTIMASI:
# 1. usecols=[0, 1, 2] -> Hanya ambil 3 kolom pertama langsung dari disk (menghemat RAM & I/O).
# 2. dtype=str -> Memaksa Pandas membaca data sebagai string tanpa menebak tipe data (menghemat CPU).
# 3. engine='c' -> Memastikan beralih ke parser bahasa C yang jauh lebih cepat dari parser Python.
df = pd.read_csv(
    DATASET_PATH,
    usecols=[0, 1, 2],
    dtype={0: str, 1: str, 2: str},
    engine='c',
    low_memory=False
)

# Menggunakan konvensi nama akademis dan memperbaiki typo koma
df.columns = ["Query", "Category", "Label"]

print(f"\n[1] DATASET DIMUAT")
print(f"    Total data mentah : {len(df)} baris")
print(f"    Kolom             : {list(df.columns)}")

print("\n[Informasi DataFrame df]")
print(f"Jumlah Baris    : {df.shape[0]} baris")
print(f"Jumlah Kolom    : {df.shape[1]} kolom")
print("\nRingkasan Informasi (df.info()):")

# JIKA Anda tetap menggunakan nama kolom 'category', gunakan baris ini:
kategori_unik = df['Category'].unique().tolist()

print("Daftar seluruh kategori yang ada:")
print(kategori_unik)
df.info()


  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES

[1] DATASET DIMUAT
    Total data mentah : 10190450 baris
    Kolom             : ['Query', 'Category', 'Label']

[Informasi DataFrame df]
Jumlah Baris    : 10190450 baris
Jumlah Kolom    : 3 kolom

Ringkasan Informasi (df.info()):
Daftar seluruh kategori yang ada:
['None_Type', 'Error-Based', 'meta_based', 'stackqueries_based', 'Time-Based', 'Union-Based', 'boolean-based']
<class 'pandas.DataFrame'>
RangeIndex: 10190450 entries, 0 to 10190449
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   Query     str  
 1   Category  str  
 2   Label     str  
dtypes: str(3)
memory usage: 233.2 MB


In [ ]:
# =============================================================================
#  TAHAP 2 — PREPROCESSING OTOMATIS (ALOKASI 3 PHYSICAL CORES)
# =============================================================================

print("\n[2] PREPROCESSING OTOMATIS (ALOKASI TERBATAS)")

sebelum = len(df)

# 1. Hapus baris dengan nilai kosong
df.dropna(subset=["Query", "Label"], inplace=True)
print(f"    Hapus baris kosong       : {sebelum - len(df)} baris dihapus")

# 2. Normalisasi format Label
df["Label"] = (
    df["Label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"yes": "1", "no": "0"})
)
df = df[df["Label"].str.match(r"^[01]$")]
df["Label"] = df["Label"].astype(int)
print(f"    Setelah filter Label     : {len(df)} baris valid")

# 3. Lowercase awal untuk menangkap duplikat (Sangat Cepat)
print("    Melakukan lowercase awal pada seluruh data mentah...")
df["Query_lowercase"] = df["Query"].astype(str).str.lower()

sebelum_lowercase_dup = len(df)
df.drop_duplicates(subset=["Query_lowercase"], inplace=True)
print(f"    Duplikat berbasis lowercase dihapus: {sebelum_lowercase_dup - len(df)} baris")

# 4. ALOKASI CORE SPESIFIK:
# Menyisakan 1 Core fisik (2 Logical Processors) untuk sistem operasi.
# Menggunakan 3 Core fisik sisanya (6 Logical Processors) untuk pemrosesan paralel.
JUMLAH_JOBS = 6 

print(f"    -> Mendeteksi {os.cpu_count()} Logical Processors di sistem.")
print(f"    -> Mengaktifkan {JUMLAH_JOBS} Logical Processors (setara 3 Core Fisik) untuk preprocessing...")

df["Query_clean"] = Parallel(n_jobs=JUMLAH_JOBS)(
    delayed(preprocess_text)(query) for query in df["Query_lowercase"]
)
print("    Preprocessing paralel selesai!")

# 5. Hapus duplikasi struktural akhir pasca-preprocessing
sebelum_struct_dup = len(df)
df.drop_duplicates(subset=["Query_clean"], inplace=True)
df = df[df["Query_clean"].str.strip() != ""]
print(f"    Duplikat struktural dihapus        : {sebelum_struct_dup - len(df)} baris")

# Hapus kolom sementara agar tidak mengotori DataFrame
df.drop(columns=["Query_lowercase"], inplace=True)

# Reset index setelah pembersihan
df.reset_index(drop=True, inplace=True)

print(f"    Total data bersih (Final)          : {len(df)} baris")
print(f"\n    Distribusi Kelas:")
distribusi = df["Label"].value_counts()
print(f"      Label 0 (Normal) : {distribusi.get(0, 0)} sampel")
print(f"      Label 1 (SQLI)   : {distribusi.get(1, 0)} sampel")


[2] PREPROCESSING OTOMATIS (ALOKASI TERBATAS)
    Hapus baris kosong       : 0 baris dihapus
    Setelah filter Label     : 10190450 baris valid
    Melakukan lowercase awal pada seluruh data mentah...
    Duplikat berbasis lowercase dihapus: 44040 baris
    -> Mendeteksi 8 Logical Processors di sistem.
    -> Mengaktifkan 6 Logical Processors (setara 3 Core Fisik) untuk preprocessing...


In [ ]:
# =============================================================================
#  TAHAP 3 — RINGKASAN HASIL PREPROCESSING
# =============================================================================

print("\n[3] TEXT PREPROCESSING SELESAI")
print("    Contoh hasil preprocessing:")
for i in range(min(3, len(df))):
    print(f"\n    [{i+1}] Asli   : {df['Query'].iloc[i][:70]}")
    print(f"         Bersih : {df['Query_clean'].iloc[i][:70]}")
    print(f"         Label  : {'SQLI (1)' if df['Label'].iloc[i] == 1 else 'Normal (0)'}")

In [ ]:
# =============================================================================
#  TAHAP tambahan — BALANCING KELAS & DOWNSAMPLING PER KATEGORI (DINAMIS)
# =============================================================================

# Tentukan target ideal jumlah baris per kategori target
TARGET_PER_KATEGORI = 15000
KATEGORI_TARGET = ['Error-Based', 'meta_based', 'stackqueries_based', 'Time-Based', 'Union-Based', 'boolean-based']

print("\n[tambahan] BALANCING & DOWNSAMPLING KELAS")

# Hitung ketersediaan baris unik asli pasca-generalisasi
jumlah_per_kat = {}
for kat in KATEGORI_TARGET:
    jumlah_per_kat[kat] = (df["Category"] == kat).sum()
    print(f"    Tersedia setelah deduplikasi struktural '{kat}': {jumlah_per_kat[kat]} baris")

# Optimasi Akademis: Menyesuaikan target sampling secara otomatis berdasarkan data terkecil agar tidak crash
min_tersedia = min(jumlah_per_kat.values())
TARGET_RIIL = min(TARGET_PER_KATEGORI, min_tersedia)
print(f"\n[Info] Target disesuaikan menjadi {TARGET_RIIL} baris per kategori SQLi untuk menghindari sample error.")

sampled_dfs = []

# Sampling masing-masing kategori target SQLi
for kat in KATEGORI_TARGET:
    df_kat = df[df["Category"] == kat].sample(n=TARGET_RIIL, random_state=42)
    sampled_dfs.append(df_kat)

# Menyeimbangkan data Normal (Label 0) dengan total akumulasi dari seluruh kategori SQLi yang diambil
TARGET_NORMAL = TARGET_RIIL * len(KATEGORI_TARGET)
df_normal = df[df["Label"] == 0].sample(n=TARGET_NORMAL, random_state=42)
sampled_dfs.append(df_normal)

# Gabungkan semua subset data dan acak urutannya (shuffle)
df_balanced = pd.concat(sampled_dfs, axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
df = df_balanced.copy()

print(f"\n    Berhasil di-balancing berdasarkan struktur query.")
print(f"    Total data gabungan setelah balancing: {len(df)} baris")

print("\n    Distribusi kategori setelah balancing:")
print(df["Category"].value_counts())

In [ ]:
# =============================================================================
#  TAHAP 4 — SPLIT DATA & ELIMINASI DATA LEAKAGE (PARALEL 3 CORES)
# =============================================================================

source_df = df_balanced if "df_balanced" in globals() else df
print(f"\n[4] SPLIT DATASET (80:20)")

df_train, df_test_raw = train_test_split(
    source_df,
    test_size=0.2,
    random_state=42,
    stratify=source_df["Label"]
)

print(f"    Data Training      : {len(df_train)} sampel")
print(f"    Data Testing Awal  : {len(df_test_raw)} sampel")
print("\n    [Proses Sensor] Memeriksa Near-Duplicate menggunakan 3 Core Fisik...")

train_raw_list = df_train["Query"].astype(str).tolist()
test_raw_list = df_test_raw["Query"].astype(str).tolist()

# Jalankan pengecekan secara paralel di 6 logical processors (3 Core)
JUMLAH_JOBS = 6
hasil_sensor = Parallel(n_jobs=JUMLAH_JOBS)(
    delayed(cek_kebocoran_per_baris)(q, train_raw_list) for q in test_raw_list
)

# Pisahkan data yang aman dan yang bocor
keep_indices = [idx for idx, bocor in enumerate(hasil_sensor) if not bocor]
count_leakage = len(test_raw_list) - len(keep_indices)

df_test = df_test_raw.iloc[keep_indices]

X_train = df_train["Query_clean"].astype(str)
y_train = df_train["Label"]
X_test = df_test["Query_clean"].astype(str)
y_test = df_test["Label"]

print(f"    Near-duplicate (leakage) ditemukan & dihapus: {count_leakage} sampel")
print(f"    Data Testing Aman (Final)                   : {len(X_test)} sampel")

In [ ]:
# =============================================================================
#  TAHAP 5 — DEFINISI PIPELINE TF-IDF + MULTINOMIAL NAÏVE BAYES
# =============================================================================

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=5,
            max_features=10000,
            sublinear_tf=True
        )
    ),
    (
        "nb",
        MultinomialNB(alpha=0.1)
    )
])

print("\n[5] PIPELINE DIDEFINISIKAN")
print("    Algoritma  : Multinomial Naïve Bayes")
print("    Ekstraksi  : TF-IDF (Char n-gram 3-5, max 10000 fitur)")
print("    Alpha (smoothing) : 0.1")


In [ ]:
# =============================================================================
#  TAHAP 6 — CROSS-VALIDATION STRATIFIED (PARALEL 3 CORES)
# =============================================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(f"\n[6] CROSS-VALIDATION (5-Fold Stratified)")
print("    Menjalankan 5-Fold Cross Validation secara paralel di 3 Core Fisik...")

# Tambahkan n_jobs=6 untuk membatasi thread komputasi
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=6 
)

print(f"    F1 per Fold : {[round(s, 4) for s in scores]}")
print(f"    Rata-rata   : {scores.mean():.4f}")
print(f"    Std Dev     : {scores.std():.4f}")

In [ ]:
# =============================================================================
#  TAHAP 7 — TRAINING MODEL
# =============================================================================

pipeline.fit(X_train, y_train)

print(f"\n[7] MODEL BERHASIL DILATIH")
print(f"    Jumlah data training : {len(X_train)} sampel")


In [ ]:
# =============================================================================
#  TAHAP 8 — EVALUASI MODEL
# =============================================================================

y_pred = pipeline.predict(X_test)

akurasi = accuracy_score(y_test, y_pred)
presisi = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\n[8] HASIL EVALUASI MODEL")
print("    " + "─" * 40)
print(f"    Akurasi   : {akurasi * 100:.2f}%")
print(f"    Presisi   : {presisi * 100:.2f}%")
print(f"    Recall    : {recall * 100:.2f}%")
print(f"    F1-Score  : {f1 * 100:.2f}%")
print("    " + "─" * 40)

tn, fp, fn, tp = cm.ravel()
print(f"\n    CONFUSION MATRIX")
print(f"    {'':20} Prediksi Normal  Prediksi SQLI")
print(f"    {'Aktual Normal':<20} {tn:<17} {fp}")
print(f"    {'Aktual SQLI':<20} {fn:<17} {tp}")
print(f"\n      TP (Benar SQLI)    : {tp}")
print(f"      TN (Benar Normal)  : {tn}")
print(f"      FP (False Positive): {fp}")
print(f"      FN (False Negative): {fn}")

print("\n    CLASSIFICATION REPORT:")
print(classification_report(
    y_test, y_pred,
    target_names=["Normal (0)", "SQLI (1)"]
))


In [ ]:
# =============================================================================
#  TAHAP 9 — UJI MANUAL DETEKSI
# =============================================================================

def deteksi_sqli_dengan_waktu(input_teks: str) -> dict:
    """Fungsi deteksi SQL Injection dengan pengukuran waktu proses."""
    start_time = time.time()
    teks_bersih = preprocess_text(input_teks)
    prediksi = pipeline.predict([teks_bersih])[0]
    probabilitas = pipeline.predict_proba([teks_bersih])[0]
    latency_ms = (time.time() - start_time) * 1000

    return {
        "input": input_teks,
        "prediksi": "SQLI" if prediksi == 1 else "Normal",
        "prob_sqli": round(probabilitas[1] * 100, 2),
        "prob_normal": round(probabilitas[0] * 100, 2),
        "status": "🚫 DIBLOKIR" if prediksi == 1 else "✅ DIIZINKAN",
        "latency_ms": round(latency_ms, 2)
    }

sampel_uji = [
    "' OR '1'='1",
    "SELECT * FROM users WHERE id = 1",
    "UNION SELECT username, password FROM users--",
    "admin",
    "1; DROP TABLE users;--",
    "search=buku",
    "-1' UNION ALL SELECT NULL,NULL,NULL--",
    "username is malik password=12345",
]

print("\n[9] UJI MANUAL DETEKSI")
print("    " + "─" * 60)

for sampel in sampel_uji:
    hasil = deteksi_sqli_dengan_waktu(sampel)
    print(f"\n    Input  : {hasil['input']}")
    print(f"    Status : {hasil['status']}")
    print(f"    P(SQLI)= {hasil['prob_sqli']}%  |  P(Normal)={hasil['prob_normal']}%")
    print(f"    Waktu  : {hasil['latency_ms']} ms  |  (Memenuhi target < 100ms)")

print("\n    " + "─" * 60)


In [ ]:
# =============================================================================
#  TAHAP 10 — SIMPAN MODEL
# =============================================================================

MODEL_PATH = "model_sqli_nb.pkl"
joblib.dump(pipeline, MODEL_PATH)

print(f"\n[8] MODEL DISIMPAN")
print(f"    Path  : {MODEL_PATH}")
print(f"    Muat kembali dengan: pipeline = joblib.load('{MODEL_PATH}')")
print("\n" + "=" * 65)
print("  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK")
print("=" * 65)